# OmniVoice Yoruba — error bars on the data-efficiency curve (T4/L4, no A100)

**Why:** nb05 produced the data-efficiency curve from a *single* eval pass per budget. Masked-diffusion
synthesis is stochastic (position/unmask order samples the global torch RNG), so one pass is noisy —
the full checkpoint alone scored `tone_i2` 0.62↔0.68 across two passes, a ~0.04 swing. The sweep's
plateau gains (5h→28.5h ≈ 0.05) are the **same size as that noise**, so the curve is not yet defensible.

This notebook re-scores **every saved checkpoint** (`0h`=zero-shot base, `1h`, `5h`, `15h`, `full`=28.5h)
on the **identical nb01 tone gate**, **×5 seeds each**, and reports `tone_i2 = mean ± std` (SEM = std/√N).
Output: an **error-barred curve** + `data_efficiency_errorbars.v1.json` on S3 — the Paper-1 headline figure,
now with error bars. Inference only; one model at a time → fits a **T4 (16 GB)**. `hf-xet` removed (load-hang gotcha).
**No edits needed — run top to bottom.**

## 1. Install + restart (numpy<2, omnivoice, gate stack; remove hf-xet)

In [ ]:
%pip install --quiet "numpy<2"
%pip install --quiet omnivoice
%pip install --quiet boto3 soundfile soxr librosa speechbrain "swift-f0" pyworld accelerate safetensors "huggingface_hub>=0.24.0" tqdm matplotlib
%pip uninstall -y hf-xet
import os
print("Installs done. RESTARTING so numpy<2 takes effect — run from the NEXT cell.", flush=True)
os._exit(0)

In [ ]:
import numpy as np
assert np.__version__.startswith("1."), "numpy<2 pin did not take — re-run install + restart"
print("numpy", np.__version__, "OK")

## 2. Clone OmniVoice (introspect API) + `mosesdaudu001/tone-on-a-budget` (approachB gate)

In [ ]:
import os, subprocess, sys, shutil
OV_DIR = "/content/OmniVoice" if os.path.isdir("/content") else "/tmp/OmniVoice"
if os.path.exists(OV_DIR): shutil.rmtree(OV_DIR)
subprocess.run(["git","clone","--depth","1","https://github.com/k2-fsa/OmniVoice.git",OV_DIR],check=True)
tsv = os.path.join(OV_DIR,"docs","lang_id_name_map.tsv"); yo_ok=False
if os.path.exists(tsv):
    for ln in open(tsv):
        c=ln.rstrip("\n").split("\t")
        if (c and c[0]=="yo") or (len(c)>1 and c[1].lower()=="yoruba"):
            print("Yoruba lang line:",ln.rstrip()); yo_ok=(c[0]=="yo")
assert yo_ok, "Yoruba code is not 'yo' — fix LANG_CODE before synthesizing"
print("OmniVoice cloned ->",OV_DIR)

In [ ]:
# tone-metric package (public) — replaces the old git clone + sys.path of tone_metric/
%pip install --quiet --no-deps "git+https://github.com/mosesdaudu001/tone-on-a-budget.git"
import os, subprocess, sys, shutil, importlib
importlib.invalidate_caches()
import tone_metric
from tone_metric import tone_probe as tp
from tone_metric import tone_f0_abs as f0a
from tone_metric.grpo_reward import RewardModels
CODE_DIR = os.path.dirname(tone_metric.__file__)   # minimal_pairs_draft.json ships as package data
print("tone_metric", tone_metric.__version__, "from", CODE_DIR)


## 3. Secrets + S3 + constants + the checkpoint POINTS + base snapshot prefetch

`POINTS` lists every checkpoint to score. `0h` loads the base OmniVoice snapshot (zero-shot); the rest pull
the sweep/full checkpoints nb05/nb04 already uploaded to `tts_checkpoints/omnivoice_yoruba/{tag}/`. The base
snapshot also supplies the text tokenizer + audio codec (`audio_tokenizer/`) that a training checkpoint omits.

In [ ]:
import os, getpass, boto3, random, torch, numpy as np, shutil, glob
def _secret(k):
    try:
        from google.colab import userdata
        v=userdata.get(k)
        if v: return v
    except Exception: pass
    return os.environ.get(k) or getpass.getpass(f"{k}: ")
os.environ["AWS_ACCESS_KEY_ID"]=_secret("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"]=_secret("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"]=os.environ.get("AWS_DEFAULT_REGION","us-east-1")
HF_TOKEN=_secret("HF_TOKEN"); os.environ["HF_TOKEN"]=HF_TOKEN or ""
if HF_TOKEN:
    from huggingface_hub import login; login(token=HF_TOKEN)
os.environ["HF_HUB_DISABLE_XET"]="1"; os.environ["HF_HUB_ENABLE_HF_TRANSFER"]="0"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"]="60"; os.environ["HF_HUB_ETAG_TIMEOUT"]="60"
try:
    import huggingface_hub.constants as _hc; _hc.HF_HUB_DISABLE_XET=True
except Exception: pass

BUCKET="codec-audio-data"
s3=boto3.client("s3",region_name=os.environ["AWS_DEFAULT_REGION"]); s3.head_bucket(Bucket=BUCKET)
print("S3 connected:",BUCKET)

DEVICE="cuda" if torch.cuda.is_available() else "cpu"
SR=24000; LANG_CODE="yo"
VARIANCE_SEEDS=[4242,1234,2026,777,99]; N_LONG=8

# every checkpoint to score: (hours, tag, s3_prefix or None=zero-shot base)
CKPT_ROOT="tts_checkpoints/omnivoice_yoruba"
POINTS=[(0.0,"base",None),
        (1.0,"1h",  f"{CKPT_ROOT}/1h"),
        (5.0,"5h",  f"{CKPT_ROOT}/5h"),
        (15.0,"15h",f"{CKPT_ROOT}/15h"),
        (28.5,"full",f"{CKPT_ROOT}/full")]

BIBLE_MANIFEST="tts_data/yoruba_gold/s1_train.v2.jsonl"
HOLDOUTS_KEY="tts_data/yoruba_gold/holdouts.v1.json"
S3_TONE_PREFIX="tts_data/yoruba/tone_v2"; F0CAL_KEY=f"{S3_TONE_PREFIX}/f0_abs_calibration.v1.json"
ABLATION_KEY="tts_data/yoruba/omnivoice_ablation/data_efficiency.v1.json"          # nb05 single-pass (overlay)
ERRBAR_KEY="tts_data/yoruba/omnivoice_ablation/data_efficiency_errorbars.v1.json"  # this notebook writes
CURVE_KEY="tts_data/yoruba/omnivoice_ablation/curve_errorbars.png"
LISTEN_PREFIX="tts_data/yoruba/omnivoice_ablation/listen"
WORK="/content/ov_errbar" if os.path.isdir("/content") else "/tmp/ov_errbar"; os.makedirs(WORK,exist_ok=True)

from huggingface_hub import snapshot_download
HUB=os.path.join(os.path.expanduser(os.environ.get("HF_HOME","~/.cache/huggingface")),"hub")
if os.path.isdir(os.path.join(HUB,".locks")): shutil.rmtree(os.path.join(HUB,".locks"),ignore_errors=True)
OV_BASE=None
for _k in range(3):
    try: OV_BASE=snapshot_download("k2-fsa/OmniVoice",max_workers=1,etag_timeout=60); break
    except Exception as e: print("base prefetch retry:",type(e).__name__,e)
assert OV_BASE and os.path.isdir(os.path.join(OV_BASE,"audio_tokenizer")), "could not pre-fetch k2-fsa/OmniVoice"
print("base snapshot:",OV_BASE,"| DEVICE",DEVICE,"| seeds",VARIANCE_SEEDS)
print("POINTS:",[(h,t) for h,t,_ in POINTS])

## 4. Gate stack (identical to nb01/nb03/nb04 §9)

In [ ]:
import json
from transformers import AutoModel, AutoFeatureExtractor
rm=RewardModels(device=DEVICE); assert rm.ecapa is not None, "ECAPA failed — SSIM needs it"
enc=AutoModel.from_pretrained("ajesujoba/AfriHuBERT").to(DEVICE).eval()
fe =AutoFeatureExtractor.from_pretrained("ajesujoba/AfriHuBERT")
objs=s3.list_objects_v2(Bucket=BUCKET,Prefix=f"{S3_TONE_PREFIX}/tone_probe_L").get("Contents")
assert objs, "no tone_probe_L* on S3 (run nb22)"
probe_key=sorted(o["Key"] for o in objs)[-1]
s3.download_file(BUCKET,probe_key,f"{WORK}/tone_probe.pt")
PROBE,PMETA=tp.load_probe(f"{WORK}/tone_probe.pt",DEVICE); LAYER=PMETA.get("layer",9)
s3.download_file(BUCKET,F0CAL_KEY,f"{WORK}/f0cal.json"); F0CAL=json.load(open(f"{WORK}/f0cal.json"))
I2_TH,I2_TL=F0CAL["theta_h"],F0CAL["theta_l"]; I2_MODE,I2_LATE=F0CAL.get("mode","blind"),F0CAL.get("late_frac",0.5)
print("probe:",probe_key,"| layer:",LAYER)

## 5. Probe set + clean bible reference (verbatim nb04 §6 — identical eval inputs)

In [ ]:
import io, json, random, soundfile as sf, numpy as np
MP=json.load(open(os.path.join(CODE_DIR,"minimal_pairs_draft.json")))
CARRIER=MP["carriers"][0]["template"]
probe_lines=[]
for s_ in MP["sets"]:
    for j,it in enumerate(s_["items"]):
        probe_lines.append(dict(id=f"mp_{s_['base']}_{j}",kind="minpair",base=s_["base"],
                                word=it["text"],tones=it.get("tones"),text=CARRIER.replace("___",it["text"])))
HOLD=json.loads(s3.get_object(Bucket=BUCKET,Key=HOLDOUTS_KEY)["Body"].read())
EVAL_ALL=list(HOLD["eval_texts"]); rng=random.Random(4242)
for k,e in enumerate(rng.sample(EVAL_ALL,N_LONG)):
    probe_lines.append(dict(id=f"long_{k}_{e['base']}",kind="long",base=e["base"],word=None,tones=None,text=e["text"]))
ref_row=None
for raw in io.TextIOWrapper(s3.get_object(Bucket=BUCKET,Key=BIBLE_MANIFEST)["Body"],encoding="utf-8"):
    r=json.loads(raw)
    if r.get("source")=="bible" and 3.0<=float(r.get("duration_sec",0))<=10.0: ref_row=r; break
assert ref_row is not None, "no bible ref row"
REF_WAV_PATH=f"{WORK}/ref.wav"; s3.download_file(BUCKET,ref_row["audio_s3_key"],REF_WAV_PATH)
REF_TEXT=ref_row["text"]; ref_wav,ref_sr=sf.read(REF_WAV_PATH,dtype="float32")
if ref_wav.ndim>1: ref_wav=ref_wav.mean(-1)
print("probe lines:",len(probe_lines),"(",sum(p['kind']=='minpair' for p in probe_lines),"minpair +",
      sum(p['kind']=='long' for p in probe_lines),"long ) | bible ref:",ref_row["audio_s3_key"])

## 6. Per-checkpoint multi-seed eval helpers

`eval_point` loads one checkpoint, runs the gate ×K seeds, frees it, returns `tone_i2 = mean ± std`. Only
`torch.manual_seed` drives `generate`, so per-seed variance is the real resynthesis noise we are quantifying.
One model resident at a time → 16 GB-safe.

In [ ]:
import os, gc, shutil, random as _random, numpy as np, soundfile as sf
import IPython.display as ipd
from collections import defaultdict
from tqdm.auto import tqdm
import torch
from omnivoice import OmniVoice

AUX=["tokenizer.json","tokenizer_config.json","special_tokens_map.json","vocab.json",
     "merges.txt","chat_template.jinja","generation_config.json","added_tokens.json"]
def s3_ls(prefix):
    out,tok=[],None
    while True:
        kw=dict(Bucket=BUCKET,Prefix=prefix)
        if tok: kw["ContinuationToken"]=tok
        r=s3.list_objects_v2(**kw); out+=[(o["Key"],o["Size"]) for o in r.get("Contents",[])]
        if r.get("IsTruncated"): tok=r.get("NextContinuationToken")
        else: break
    return out
def materialize_ckpt(local_dir):
    have=set(os.listdir(local_dir))
    for fn in AUX:
        if fn not in have and os.path.exists(os.path.join(OV_BASE,fn)):
            shutil.copy(os.path.join(OV_BASE,fn),os.path.join(local_dir,fn))
    at=os.path.join(local_dir,"audio_tokenizer")
    if not os.path.isdir(at): shutil.copytree(os.path.join(OV_BASE,"audio_tokenizer"),at)
    return local_dir
def pull_top_level(prefix,dest):
    os.makedirs(dest,exist_ok=True); n=0
    for k,sz in s3_ls(prefix+"/"):
        tail=k[len(prefix)+1:]
        if not tail or "/" in tail: continue        # skip folder markers + subdir files
        s3.download_file(BUCKET,k,os.path.join(dest,os.path.basename(k))); n+=1
    return n

def _bal_tone_acc(pairs):
    if not pairs: return float("nan")
    tot,cor=defaultdict(int),defaultdict(int)
    for pp,tt in pairs: tot[tt]+=1; cor[tt]+=int(pp==tt)
    recs=[cor[c]/tot[c] for c in tot if tot[c]>0]
    return float(np.mean(recs)) if recs else float("nan")
def score_clip(wav,fs,text):
    logits,n16=rm.asr_logits(wav,fs)
    cer=RewardModels.cer(text,rm.decode_logits(logits))
    i1=tp.probe_score(wav,fs,text,PROBE,enc,fe,asr=rm.asr,proc=rm.asr_proc,device=DEVICE,
                      emissions=logits,n16=n16,layer=LAYER)
    i2=f0a.f0_abs_score(wav,fs,text,asr=rm.asr,proc=rm.asr_proc,device=DEVICE,emissions=logits,n16=n16,
                        theta_h=I2_TH,theta_l=I2_TL,mode=I2_MODE,mid_ref=None,late_frac=I2_LATE)
    pairs=[(pp,tt) for pp,tt in zip(i2["pred"],i2["target"]) if pp is not None]
    return dict(cer=cer,i1_acc=i1["accuracy"],i1_cov=i1["coverage"],
                tone_i2_cov=i2["coverage"],ssim=rm.ssim(wav,fs,ref_wav,ref_sr),
                len_ratio=(len(wav)/fs)/max(0.157*len(text),1e-6),pairs=pairs)
def one_pass(m,seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    np.random.seed(seed); _random.seed(seed)
    rows=[]
    for p in probe_lines:
        a=m.generate(text=p["text"],language=LANG_CODE,ref_audio=REF_WAV_PATH,ref_text=REF_TEXT)
        rows.append(dict(kind=p["kind"],**score_clip(np.asarray(a[0],dtype="float32"),SR,p["text"])))
    allp=[pr for r in rows for pr in r["pairs"]]; v=lambda k:[r[k] for r in rows if r[k]==r[k]]
    return dict(seed=seed,tone_i2=_bal_tone_acc(allp),cer=float(np.mean(v("cer"))),
                ssim=float(np.mean(v("ssim"))),i1_acc=float(np.mean(v("i1_acc"))),
                tone_i2_cov=float(np.mean([r["tone_i2_cov"] for r in rows])),
                len_ratio=float(np.mean([r["len_ratio"] for r in rows])))

KEYS=["tone_i2","cer","ssim","i1_acc","tone_i2_cov","len_ratio"]
def _stat(runs,k):
    xs=[r[k] for r in runs]; N=len(xs)
    return dict(mean=float(np.mean(xs)),std=float(np.std(xs)),sem=float(np.std(xs)/max(N**0.5,1)),
                min=float(np.min(xs)),max=float(np.max(xs)))
def eval_point(hours,tag,prefix):
    if prefix is None:
        local_dir=OV_BASE                              # zero-shot base
    else:
        local_dir=os.path.join(WORK,tag)
        n=pull_top_level(prefix,local_dir); assert n>0, f"nothing under s3://{BUCKET}/{prefix}/ — was it uploaded?"
        materialize_ckpt(local_dir)
        f=set(os.listdir(local_dir))
        assert "config.json" in f and any(x.endswith((".safetensors",".bin",".pt")) for x in f), f"no model in {tag}: {sorted(f)}"
    m=OmniVoice.from_pretrained(local_dir,device_map=("cuda:0" if DEVICE=="cuda" else "cpu"),dtype=torch.float16)
    runs=[one_pass(m,s) for s in tqdm(VARIANCE_SEEDS,desc=f"{tag} seeds")]
    st={k:_stat(runs,k) for k in KEYS}
    # inline listen + upload one representative long clip (feedback: always play eval audio)
    torch.manual_seed(VARIANCE_SEEDS[0])
    _long=[p for p in probe_lines if p["kind"]=="long"] or probe_lines
    _a=m.generate(text=_long[0]["text"],language=LANG_CODE,ref_audio=REF_WAV_PATH,ref_text=REF_TEXT)
    _w=np.asarray(_a[0],dtype="float32"); _loc=f"{WORK}/listen_{tag}.wav"; sf.write(_loc,_w,SR)
    s3.upload_file(_loc,BUCKET,f"{LISTEN_PREFIX}/{tag}.wav")
    print(f"  [{tag}] {hours:>5.1f}h  tone_i2 {st['tone_i2']['mean']:.3f} ± {st['tone_i2']['std']:.3f} "
          f"(sem {st['tone_i2']['sem']:.3f})  cer {st['cer']['mean']:.3f}  ssim {st['ssim']['mean']:.3f}   listen ({_long[0]['text'][:42]}):")
    ipd.display(ipd.Audio(_w,rate=SR))
    del m; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return dict(hours=hours,tag=tag,n_seeds=len(runs),per_pass=runs,stats=st)
print("helpers defined — run §7 to score every checkpoint.")

## 7. Run every checkpoint ×5 seeds → error bars (writes `data_efficiency_errorbars.v1.json`)

In [ ]:
import json, datetime
RESULTS=[eval_point(h,t,p) for (h,t,p) in POINTS]
print("\n=== data efficiency with error bars (tone_i2 = mean ± std over",len(VARIANCE_SEEDS),"seeds) ===")
print(f"{'hours':>7} {'tag':>6}   {'tone_i2 (mean±std)':>20}  {'cer':>6}  {'ssim':>6}  {'cov':>5}")
for r in RESULTS:
    s=r["stats"]
    print(f"{r['hours']:>7.1f} {r['tag']:>6}   {s['tone_i2']['mean']:.3f} ± {s['tone_i2']['std']:.3f}        "
          f"{s['cer']['mean']:>6.3f}  {s['ssim']['mean']:>6.3f}  {s['tone_i2_cov']['mean']:>5.2f}")
print("\nAnchors: zero-shot 0.598 | native ~0.58 | chance 0.333  (tone_i2)")
OUT=dict(date=datetime.date.today().isoformat(),seeds=VARIANCE_SEEDS,
         protocol=f"per-checkpoint eval variance over {len(VARIANCE_SEEDS)} seeds; identical nb01 tone gate; "
                  f"n_probe={len(probe_lines)}",
         anchors=dict(zero_shot=0.598,native=0.58,chance=1.0/3),
         points=[dict(hours=r["hours"],tag=r["tag"],n_seeds=r["n_seeds"],stats=r["stats"],per_pass=r["per_pass"])
                 for r in RESULTS])
s3.put_object(Bucket=BUCKET,Key=ERRBAR_KEY,Body=json.dumps(OUT,ensure_ascii=False,indent=2).encode())
print("\n-> s3://%s/%s"%(BUCKET,ERRBAR_KEY))
print("   listen wavs -> s3://%s/%s/{base,1h,5h,15h,full}.wav"%(BUCKET,LISTEN_PREFIX))

## 8. Error-barred curve (overlays nb05 single-pass points) → `curve_errorbars.png`

In [ ]:
import numpy as np, matplotlib.pyplot as plt, json
xs=[r["hours"] for r in RESULTS]
ys=[r["stats"]["tone_i2"]["mean"] for r in RESULTS]
es=[r["stats"]["tone_i2"]["std"] for r in RESULTS]
plt.figure(figsize=(6.8,4.4))
plt.errorbar(xs,ys,yerr=es,fmt="o-",capsize=4,lw=2,label=f"finetune (mean ± std, {len(VARIANCE_SEEDS)} seeds)")
try:                                                  # overlay nb05's single noisy pass for context
    SP=json.loads(s3.get_object(Bucket=BUCKET,Key=ABLATION_KEY)["Body"].read())
    sx=[v["train_hours_actual"] for v in SP["runs"].values()]; sy=[v["tone_i2"] for v in SP["runs"].values()]
    plt.scatter(sx,sy,marker="x",c="grey",alpha=0.8,zorder=3,label="single-pass (nb05 sweep)")
except Exception as e: print("no single-pass overlay:",e)
plt.axhline(1.0/3,ls=":",c="grey",label="chance 0.333")
plt.axhline(0.58,ls="--",c="green",label="native ~0.58")
for r in RESULTS:
    if r["tag"] in ("base","full"):
        plt.annotate(r["tag"],(r["hours"],r["stats"]["tone_i2"]["mean"]),
                     textcoords="offset points",xytext=(5,6),fontsize=9)
plt.xlabel("clean Yoruba hours"); plt.ylabel("tone_i2 (balanced over TBUs)")
plt.title("Data efficiency: lexical tone vs target-language hours")
plt.legend(fontsize=8); plt.grid(alpha=0.25); plt.tight_layout()
plt.savefig(f"{WORK}/curve_errorbars.png",dpi=130)
s3.upload_file(f"{WORK}/curve_errorbars.png",BUCKET,CURVE_KEY)
plt.show()
print("-> s3://%s/%s"%(BUCKET,CURVE_KEY))

### Read-out
- **Headline:** `tone_i2 = mean ± std` per hours point. If the error bars at `5h/15h/full` separate from
  `base` (0.598) and from each other, the install-by-~5h-then-plateau story is real; if they overlap, the
  honest claim is **"tone installs to its plateau by ~5 h and does not improve thereafter"** (a *stronger*,
  cleaner data-efficiency result than a noisy upward drift).
- **The 1h dip** (single-pass 0.545 < zero-shot 0.598): check whether it survives the error bar — if `1h`
  sits below `base` by more than the combined spread, "too little data perturbs pretrained tone before it
  installs the target system" is a real, publishable finding; if it's within noise, drop the claim.
- **CER stays low everywhere** (intelligibility never regresses) — pairs with nb06's "CER is tone-blind".
- **Listen** (`…/omnivoice_ablation/listen/{tag}.wav`) is the decisive check the metric isn't being gamed.
- Artifacts: `data_efficiency_errorbars.v1.json` + `curve_errorbars.png` on S3 → drops straight into Paper-1.